In [0]:
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark.conf.set("spark.sql.execution.arrow.pyspark.maxRecordsPerBatch", 1)
#spark.conf.set("spark.task.resource.memory", "4g")
#spark.conf.set("spark.sql.files.maxPartitionBytes", 1 * 1024 * 1024)
#spark.conf.set("spark.task.resource.myresource.amount", "1")
#spark.conf.set("spark.executor.resource.myresource.amount", "1")
#spark.conf.get("spark.speculation")

In [0]:
dbutils.widgets.text("input_folder", "Hackathon2/", "Input DICOM Folder")
dbutils.widgets.text("output_parquet", "Hackathon2_output", "Output Parquet")
dbutils.widgets.text("file_size_threshold", "5242880", "File size threshold (bytes)")
dbutils.widgets.text("batch_size", "1000", "Batch Size")
dbutils.widgets.text("sampling_size", "0", "Sampling Size (0=disabled)")


In [0]:
for widget in ["input_folder", "output_parquet", "file_size_threshold", "batch_size", "sampling_size"]:
    print(f"{widget}: {dbutils.widgets.get(widget)}")

In [0]:
import pydicom
from tqdm import tqdm
#import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import os
from io import BytesIO
from pydicom.filebase import DicomFileLike
from openai import OpenAI
from PIL import Image
import base64



In [0]:
input_dir = f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get("input_folder")}"


def list_dcm_files(folder):
    files = os.listdir(folder)
    files = [x for x in files if x.endswith(".dcm")  and "DICOMDIR" not in x]
    return files


df_all = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .option("includeBinaryFiles", "false")
    .load(input_dir)
    .select("path", "length")
    #.select("path", "length", "content")
)
display(df_all.limit(10))
print(df_all.count())

In [0]:
FILE_LENGTH_THRES = int(dbutils.widgets.get("file_size_threshold"))
df_large = df_all.filter(f"length > {FILE_LENGTH_THRES}")

large_file_output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('output_parquet'))}_large_files.parquet"
df_large.write.mode("overwrite").parquet(large_file_output_path)

#print(df_large.count())
df_small = df_all.filter(f"length <= {FILE_LENGTH_THRES}")
print(df_small.count())

In [0]:
df = df_small

In [0]:
from pyspark.sql.functions import regexp_extract, lit
#df = df.withColumn("parent_dir_path", regexp_extract("path", r"(.*/)[^/]+\.dcm$", 1))
#df = df.withColumn("grandparent_dir_path", regexp_extract("parent_dir_path", r"(.*/)[^/]+/$", 1))

df = df.withColumn("dcm_int", regexp_extract("path", r"(\d+)\.dcm$", 1).cast("int"))

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

from pyspark.sql.functions import rand
SAMPLING_SIZE = int(dbutils.widgets.get("sampling_size"))


from pyspark.sql.functions import monotonically_increasing_id

if SAMPLING_SIZE > 0:
    #df2 = df2.withColumn("row_idx", monotonically_increasing_id())
    #display(df.filter(f"dcm_int <= {SAMPLING_SIZE}").limit(1000))
    #print(df2.count())
    df = df.filter(f"dcm_int <= {SAMPLING_SIZE}")
    print(df.count())


In [0]:

import math
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

MAX_BATCH = math.ceil(float(df.count())/float(BATCH_SIZE))
print("batch size:", BATCH_SIZE)
print("max batch:", MAX_BATCH)
from pyspark.sql.functions import floor

df = df.withColumn("batch_idx", floor(rand() * MAX_BATCH).cast("int"))

In [0]:
display(df.limit(1000))

In [0]:
intmd_output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('output_parquet'))}_intermediate.parquet"
df.write.mode("overwrite").partitionBy("batch_idx").parquet(intmd_output_path)

In [0]:
import time
import pandas as pd
from pathlib import Path
from pydicom.encaps import get_frame, generate_pixel_data_frame
import gc

api_key = dbutils.secrets.get(scope = "adc_store", key = "image-anon_gpt41mini_api-key")


def extract_dicom_tags(file_path):
    ds_header = pydicom.dcmread(file_path, stop_before_pixels=True)
    accession_number = ds_header.get("AccessionNumber", None)
    modality = ds_header.get("Modality", None)
    return accession_number, modality


def extract_first_frame(file_path):
    """Extract frame 0 from any DICOM. Memory-safe for multi-frame files."""
    # Read header to check frame count
    ds_header = pydicom.dcmread(file_path, stop_before_pixels=True)
    rows = ds_header.Rows
    cols = ds_header.Columns
    bits = ds_header.BitsAllocated
    num_frames = int(ds_header.NumberOfFrames) if "NumberOfFrames" in ds_header else 1
    samples = ds_header.SamplesPerPixel if "SamplesPerPixel" in ds_header else 1

    del ds_header

    # Estimate full uncompressed size
    full_size_mb = rows * cols * (bits // 8) * num_frames * samples / (1024**2)

    ds = pydicom.dcmread(file_path, stop_before_pixels=False)

    if num_frames > 1 and full_size_mb > 50:
        # Large multi-frame: extract only first compressed frame
        is_encapsulated = (
            hasattr(ds["PixelData"], "is_undefined_length")
            and ds["PixelData"].is_undefined_length
        )
        if is_encapsulated:
            try:
                frame_bytes = get_frame(ds.PixelData, 0)
            except Exception:
                frame_bytes = next(generate_pixel_data_frame(ds.PixelData, num_frames))
            del ds
            gc.collect()
            return np.array(Image.open(BytesIO(frame_bytes)))
        else:
            # Uncompressed multi-frame: slice raw bytes for frame 0
            frame_nbytes = rows * cols * (bits // 8) * samples
            raw = ds.PixelData[:frame_nbytes]
            del ds
            dtype = np.uint8 if bits == 8 else np.uint16
            return np.frombuffer(raw, dtype=dtype).reshape(rows, cols)
    else:
        # Single frame or small multi-frame: safe to decompress all
        arr = ds.pixel_array
        if arr.ndim == 3 and num_frames > 1:
            frame = arr[0].copy()
            del arr
        else:
            frame = arr
        del ds
        return frame
    
def frame_to_b64(frame_arr):
    """Resize frame to 512x512 and encode as base64 PNG."""
    img = Image.fromarray(frame_arr)
    del frame_arr
    img = img.resize((512, 512), Image.LANCZOS)

    arr_small = np.array(img, dtype=np.float32)
    del img
    arr_min, arr_max = arr_small.min(), arr_small.max()
    if arr_max > arr_min:
        arr_small = ((arr_small - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
    else:
        arr_small = np.zeros(arr_small.shape, dtype=np.uint8)

    img_out = Image.fromarray(arr_small)
    del arr_small
    buf = BytesIO()
    img_out.save(buf, format="PNG", optimize=True)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    buf.close()
    del img_out
    return b64

def llm_pii_detection_batch(pdf_iter):
    system_prompt = "Output the result in a structured format with 4 required fields: is_medical_scan, is_text_detected, is_personal_information_detected (e.g. name, ID numbers), and confidence_score. Valid values for is_medical_scan are yes, no, and not_sure. Answer yes only if the input image is a typical medical scan of an anatomical structure. Answer no if the input image is a screen capture, graph, or document etc. Valid values for is_text_detected are yes, no, and not_sure. Valid values for is_personal_information_detected are yes, no, and not_sure. confidence_score is an integer from 0 to 100. 0 indicates high confidence of no text detected. 100 indicates high confidence of text being detected. 50 indicates uncertain. Do not leave any field empty. Always respond with 4 fields everytime."

    user_prompt = "Tell me if there is any text in the image. Also tell me if these texts contain any personal information e.g. name, date of birth, scan date, ID numbers. Then give me a confidence score."

    deployment_name = "gpt-4.1-mini"

    endpoint = "https://image-anonymisation-resource.cognitiveservices.azure.com/openai/v1/"



    client = OpenAI(base_url=f"{endpoint}",api_key=api_key)

    for pdf in pdf_iter:
        redacted_rows = []
        #for path, content in zip(pdf['path'], pdf['content']):
        for path in pdf['path']:
            try:
                start_time_str = time.strftime("%Y-%m-%d %H:%M:%S")
                is_loaded_ok = False
                accession_number = None
                modality = None

                '''
                #ds = pydicom.dcmread(BytesIO(content), force=True)
                ds = pydicom.dcmread(path[5:])
                #ds.decompress()


                # Extract pixel array
                pixel_array = ds.pixel_array.astype(np.float32)

                # Normalize to 0-255
                pixel_array -= pixel_array.min()
                pixel_array /= pixel_array.max()
                pixel_array *= 255.0
                pixel_array = pixel_array.astype(np.uint8)

                # Convert to PIL image
                image = Image.fromarray(pixel_array)

                # Save PNG to memory buffer
                buffer = BytesIO()
                image.save(buffer, format="PNG")

                # Convert to Base64
                base64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
                '''

                #ds = pydicom.dcmread(path[5:], stop_before_pixels=False)

                # Get pixel data (no extra copy yet)
                #arr = ds.pixel_array  # already numpy
                path = path[5:]
                frame = extract_first_frame(path)
                base64_str = frame_to_b64(frame)
                del frame
                '''
                arr = extract_first_frame(path)

                # Normalize WITHOUT converting to float32 full-size
                arr_min = arr.min()
                arr_max = arr.max()

                if arr_max > arr_min:
                    # Use float32 temporarily but avoid multiple copies
                    arr = ((arr - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
                else:
                    arr = np.zeros(arr.shape, dtype=np.uint8)

                # Convert directly to image
                image = Image.fromarray(arr)

                # Encode directly to base64 (avoid buffer.getvalue() copy)
                buffer = BytesIO()
                image.save(buffer, format="PNG", optimize=True)

                base64_str = base64.b64encode(buffer.getbuffer()).decode("utf-8")

                # Explicit cleanup (important in UDFs)
                buffer.close()
                del arr, image
                '''
                is_loaded_ok = True




                completion = client.chat.completions.create(
                    model=deployment_name,
                    messages=[
                        {
                            "role": "system",
                            "content": system_prompt,
                        },
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text", "text": user_prompt
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/png;base64,{base64_str}"}
                                }

                            ]
                        }
                    ]
                )

                result = completion.choices[0].message.content
                accession_number, modality = extract_dicom_tags(path)
                
                redacted_rows.append((path, is_loaded_ok, result,  accession_number, modality, None, start_time_str, time.strftime("%Y-%m-%d %H:%M:%S")))

            except Exception as e:
                redacted_rows.append((path, is_loaded_ok, None, accession_number, modality, str(e)[:1000], start_time_str, time.strftime("%Y-%m-%d %H:%M:%S")))


        yield pd.DataFrame(redacted_rows, columns=["path", "is_loaded_ok", "llm_result", "accession_number", "modality", "error_message", "start_ts", "end_ts"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType, BooleanType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("llm_result", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("modality", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("start_ts", StringType(), True),
    StructField("end_ts", StringType(), True)
])



In [0]:
import math
from tqdm.notebook import tqdm

#BATCH_SIZE = 100
#total_batch_count = math.ceil(df2.count() / BATCH_SIZE)

intmd_df = spark.read.parquet(intmd_output_path)

for batch_idx in tqdm(range(MAX_BATCH), desc="Batches"):

    # Run mapInPandas redaction
    pii_detect_df = intmd_df.filter(f"batch_idx = {batch_idx}").mapInPandas( llm_pii_detection_batch, schema=output_schema)

    #results = pii_detect_df.collect()
    #pii_detect_df = pii_detect_df.repartition(pii_detect_df.count())
    output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('output_parquet'))}_batch{batch_idx}.parquet"

    pii_detect_df.write.mode("overwrite").parquet(output_path)


In [0]:
batch_idx = 0
output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('output_parquet'))}_batch{batch_idx}.parquet"
df_out = spark.read.parquet(output_path)#.filter("llm_result is not null")
display(df_out.limit(1000))
print(df_out.count())

In [0]:
batch_idx = 1
output_path = f"/Volumes/1_inland/sectra/vone/{str(dbutils.widgets.get('output_parquet'))}_batch{batch_idx}.parquet"
df_out = spark.read.parquet(output_path)#.filter("llm_result is not null")
from pyspark.sql.functions import regexp_extract, col

from pyspark.sql.functions import from_json, schema_of_json

# Infer schema from a sample row
sample_json = df_out.select("llm_result").filter(col("llm_result").isNotNull()).limit(1).collect()[0]["llm_result"]
json_schema = schema_of_json(sample_json)

df_out = df_out.withColumn(
    "llm_result_json", from_json(col("llm_result"), json_schema)
#).withColumn(
#    "is_text_detected", col("llm_result_json.is_medical_scan")
).withColumn(
    "is_text_detected", col("llm_result_json.is_text_detected")
).withColumn(
    "is_personal_information_detected", col("llm_result_json.is_personal_information_detected")
).withColumn(
    "confidence_score", col("llm_result_json.confidence_score")
)

display(df_out)

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

display(
    df_out.groupBy("modality")
    .agg(
        #spark_sum(when(col("is_medical_scan") == "yes", 1).otherwise(0)).alias("yes_medical_scan"),
        #spark_sum(when(col("is_medical_scan") == "no", 1).otherwise(0)).alias("no_medical_scan"),
        spark_sum(when(col("is_text_detected") == "yes", 1).otherwise(0)).alias("yes_text_count"),
        spark_sum(when(col("is_text_detected") == "no", 1).otherwise(0)).alias("no_text_count"),
        spark_sum(when(col("is_personal_information_detected") == "yes", 1).otherwise(0)).alias("yes_personal_info_count"),
        spark_sum(when(col("is_personal_information_detected") == "no", 1).otherwise(0)).alias("no_personal_info_count")
    )
    .orderBy("modality")
)

In [0]:
display(df_out.filter("modality = 'XA' AND is_personal_information_detected = 'yes'"))